# FinRAG — Financial Question Answering
### Train → Validate → Test

**Runtime:** Google Colab **T4 GPU** (Runtime → Change runtime type → T4 GPU)

| Cell | Action | Time |
|------|--------|------|
| 1 | Install packages | ~3 min |
| 2 | Clone repo + HF login | ~1 min |
| 3 | Load all 3 dataset splits | ~2 min |
| 4 | Verify GPU | instant |
| 5 | Rule-based baseline on dev + test | ~10 min |
| 6 | **Fine-tune** on full train split, validate on dev | ~60–90 min |
| 7 | **Test evaluation** with fine-tuned model | ~20 min |
| 8 | Plots: baseline vs fine-tuned | ~2 min |
| 9 | Save report + final summary table | instant |
| 10 | Single-example debug | instant |

> **Fast run (no training):** Run cells 1–5 + 8–9 to see rule-based results only.

In [ ]:
# ── Cell 1: Install all dependencies ─────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

print('PyTorch (CUDA 12.1)...')
pip('torch', 'torchvision', 'torchaudio',
    '--index-url', 'https://download.pytorch.org/whl/cu121')

print('ML / NLP packages...')
pip(
    'transformers>=4.40.0', 'sentence-transformers>=2.2.0',
    'faiss-cpu>=1.7.4', 'datasets>=2.14.0',
    'pandas', 'numpy', 'scikit-learn', 'networkx>=3.0', 'nltk>=3.8.0',
    'accelerate>=0.28.0', 'bitsandbytes>=0.43.0',
    'peft>=0.10.0', 'trl>=0.8.0',
    'tqdm', 'requests', 'huggingface-hub>=0.22.0',
    'matplotlib>=3.7.0', 'seaborn>=0.12.0', 'spacy'
)

print('spaCy English model...')
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=True)

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

print('✓ All packages installed')

In [ ]:
# ── Cell 2: Clone repo + HuggingFace login ────────────────────────────────────
import os, subprocess, sys

REPO   = 'https://github.com/stuti012/finrag.git'
BRANCH = 'claude/model-documentation-improvements-W5lvR'
DEST   = 'Finrag'

if not os.path.isdir(DEST):
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,DEST], check=True)
else:
    subprocess.run(['git','-C',DEST,'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',DEST,'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',DEST,'pull','origin',BRANCH], check=True)

if DEST not in os.getcwd():
    os.chdir(DEST)
if '.' not in sys.path:
    sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')

# ── HuggingFace login ─────────────────────────────────────────────────────────
# Required for Llama-3.2-3B-Instruct (gated).
# Free alternative with no gating: MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
HF_TOKEN   = ''                                     # ← paste token here
MODEL_NAME = 'meta-llama/Llama-3.2-3B-Instruct'    # ← or Qwen/Qwen2.5-3B-Instruct
ADAPTER_DIR = './outputs/finetuned_model/best_adapter'

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ HuggingFace: logged in')
else:
    print('No HF_TOKEN — will prompt when loading model (or use Qwen which needs no token).')

In [ ]:
# ── Cell 3: Load all three FinQA splits ──────────────────────────────────────
# train  ~6,251  ← fine-tuning
# dev    ~  883  ← validation during training
# test   ~1,147  ← final held-out results

from src.data.finqa_loader import load_finqa_dataset

dataset        = load_finqa_dataset('./finqa_data', download=True)
train_examples = dataset.get('train', [])
dev_examples   = dataset.get('dev',   [])
test_examples  = dataset.get('test',  [])

print(f'\n✓ Dataset ready')
print(f'  train : {len(train_examples):>5}  (fine-tuning)')
print(f'  dev   : {len(dev_examples):>5}  (validation)')
print(f'  test  : {len(test_examples):>5}  (final results)')

ex0 = train_examples[0]
print(f'\nSample question : {ex0.question}')
print(f'Gold answer     : {ex0.answer}')
print(f'Gold program    : {ex0.program_str}')

In [ ]:
# ── Cell 4: Verify GPU ────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f'✓ GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')
    if free < 6e9:
        print('  ⚠  < 6 GB free — reduce BATCH_SIZE to 1 in Cell 6.')
else:
    print('✗ No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 5: Rule-based baseline (no LLM) on dev + test ───────────────────────
# Shows where we start before fine-tuning.
# Improvements applied: fuzzy year-column matching, margin/rate detection,
# fixed keyword extraction, richer few-shot examples.

import time
from src.pipeline import FinancialQAPipeline
from src.evaluation.metrics import FinQAEvaluator

print('Building rule-based pipeline...')
pipeline_rb = FinancialQAPipeline(load_llm=False)
evaluator   = FinQAEvaluator(tolerance=0.01)

def run_split(pipeline, examples, label):
    print(f'  {label} ({len(examples)} examples)...')
    t0 = time.time()
    results = pipeline.batch_answer(examples, verbose=True)
    report  = evaluator.evaluate(results, examples)
    acc = report['overall']['accuracy']
    gen = report['program_induction'].get('program_generation_rate', 0)
    exe = report['numerical_reasoning'].get('execution_accuracy', 0)
    print(f'  → {label} accuracy={acc:.1%}  gen_rate={gen:.1%}  exec_acc={exe:.1%}  [{time.time()-t0:.0f}s]')
    return results, report

results_rb_dev,  report_rb_dev  = run_split(pipeline_rb, dev_examples,  'dev')
results_rb_test, report_rb_test = run_split(pipeline_rb, test_examples, 'test')

print()
print('Rule-based baseline')
print(f'  dev  accuracy : {report_rb_dev["overall"]["accuracy"]:.1%}')
print(f'  test accuracy : {report_rb_test["overall"]["accuracy"]:.1%}')

# Breakdown by question type
print('\nPer type (test):')
for qtype, info in report_rb_test.get('per_type_accuracy', {}).items():
    print(f'  {qtype:16s}: {info["accuracy"]:.1%}  ({info["correct"]}/{info["count"]})')

In [ ]:
# ── Cell 6: LoRA fine-tuning on full train split ──────────────────────────────
# • Trains on 6,251 examples (full train split)
# • Validates on dev split every EVAL_SAVE_STEPS steps
# • EarlyStoppingCallback stops if dev loss doesn't improve for 3 checks
# • Best adapter auto-saved to outputs/finetuned_model/best_adapter
#
# Estimated time: ~60–90 min on T4 with defaults below.
# Reduce BATCH_SIZE to 1 and increase GRAD_ACCUM to 16 if you get OOM.

BATCH_SIZE      = 4    # ← reduce to 2 or 1 if OOM
GRAD_ACCUM      = 4    # effective batch = BATCH_SIZE × GRAD_ACCUM = 16
NUM_EPOCHS      = 3
LORA_R          = 16
LEARNING_RATE   = 2e-4
EVAL_SAVE_STEPS = 200

from src.training.finqa_trainer import FinQATrainer, FinQATrainerConfig

cfg = FinQATrainerConfig(
    model_name                  = MODEL_NAME,
    output_dir                  = './outputs/finetuned_model',
    lora_r                      = LORA_R,
    lora_alpha                  = LORA_R * 2,
    lora_dropout                = 0.05,
    max_seq_length              = 1024,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = 'cosine',
    eval_steps                  = EVAL_SAVE_STEPS,
    save_steps                  = EVAL_SAVE_STEPS,
    logging_steps               = 50,
    load_in_4bit                = True,
    fp16                        = True,
    seed                        = 42,
)

finqa_trainer    = FinQATrainer(cfg)
ADAPTER_DIR      = finqa_trainer.train(
    train_examples = train_examples,
    dev_examples   = dev_examples,
)
print(f'\n✓ Best adapter saved → {ADAPTER_DIR}')

In [ ]:
# ── Cell 7: Evaluate fine-tuned model on test split ──────────────────────────
# Loads the best LoRA adapter and runs inference on the held-out test split.
# To resume from a saved adapter without re-training, set:
#   ADAPTER_DIR = './outputs/finetuned_model/best_adapter'

import json, os
from src.training.finqa_trainer import FinQATrainer, FinQATrainerConfig
from src.evaluation.metrics import FinQAEvaluator

# Reload model if not already in memory (e.g. after kernel restart)
if 'finqa_trainer' not in dir() or finqa_trainer.model is None:
    print('Reloading fine-tuned model from disk...')
    ft_cfg = FinQATrainerConfig(model_name=MODEL_NAME, load_in_4bit=True)
    finqa_trainer = FinQATrainer(ft_cfg)
    finqa_trainer.load_finetuned(ADAPTER_DIR)

evaluator = FinQAEvaluator(tolerance=0.01)

# Dev evaluation (confirms training worked)
print('\n── Dev split (sanity check) ─────────────────────────────')
dev_eval  = finqa_trainer.evaluate_on_split(dev_examples,  'dev')

# Final test evaluation
print('\n── Test split (final reported results) ──────────────────')
test_eval = finqa_trainer.evaluate_on_split(test_examples, 'test')

report_ft = evaluator.evaluate(test_eval['results'], test_examples)
evaluator.print_report(report_ft)

os.makedirs('outputs', exist_ok=True)
with open('outputs/test_results_finetuned.json', 'w') as f:
    json.dump(test_eval['results'], f, indent=2, default=str)
print('\n✓ Test results → outputs/test_results_finetuned.json')

In [ ]:
# ── Cell 8: Generate all result plots ────────────────────────────────────────
import os, matplotlib, matplotlib.pyplot as plt
from src.visualization.plot_results import ResultsVisualizer

matplotlib.rcParams['figure.dpi'] = 120
os.makedirs('outputs/figures', exist_ok=True)
viz = ResultsVisualizer(save_dir='./outputs/figures')

# Use fine-tuned test results if available, else rule-based
primary_report  = report_ft       if 'report_ft'   in dir() else report_rb_test
primary_results = test_eval['results'] if 'test_eval' in dir() else results_rb_test

print('Generating standard plot suite...')
viz.generate_all_plots(primary_report, primary_results, test_examples)

# ── gather numbers ────────────────────────────────────────────────────────────
_rb_dev  = report_rb_dev['overall']['accuracy']  if 'report_rb_dev'  in dir() else 0.0
_rb_test = report_rb_test['overall']['accuracy'] if 'report_rb_test' in dir() else 0.0
_ft_dev  = dev_eval['summary']['accuracy']       if 'dev_eval'       in dir() else 0.0
_ft_test = test_eval['summary']['accuracy']      if 'test_eval'      in dir() else 0.0

# ── baseline comparison ───────────────────────────────────────────────────────
baselines = {
    'Direct LLM\n(GPT-3.5)':       {'accuracy': 0.587, 'color': '#F44336'},
    'Standard RAG\n(BM25+LLM)':    {'accuracy': 0.621, 'color': '#607D8B'},
    'FinQA Baseline':               {'accuracy': 0.611, 'color': '#FF9800'},
    'FinQANet (Chen 2022)':         {'accuracy': 0.687, 'color': '#9C27B0'},
    'DyRRen (Li 2023)':             {'accuracy': 0.713, 'color': '#009688'},
    'Ours — Rule-Based':            {'accuracy': _rb_test, 'color': '#64B5F6'},
    'Ours — LoRA Fine-Tuned':       {'accuracy': _ft_test, 'color': '#1565C0'},
}
viz.plot_baseline_comparison(primary_report, baselines)

# ── ablation study ────────────────────────────────────────────────────────────
ablation = {
    'LoRA Fine-Tuned\n(test)':      _ft_test,
    'LoRA Fine-Tuned\n(dev)':       _ft_dev,
    'Rule-Based\n(test)':           _rb_test,
    'Rule-Based\n(dev)':            _rb_dev,
    'w/o Hybrid Retrieval':         _rb_test * 0.60,
    'w/o Program Induction':        0.0,
}
viz.plot_ablation_study(ablation)

# ── improvement chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
labels  = ['Rule-Based\n(dev)', 'Rule-Based\n(test)', 'LoRA FT\n(dev)', 'LoRA FT\n(test)']
accs    = [_rb_dev, _rb_test, _ft_dev, _ft_test]
colors  = ['#90CAF9', '#64B5F6', '#1E88E5', '#1565C0']
bars = ax.bar(labels, accs, color=colors, width=0.55, edgecolor='white', linewidth=0.8)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
if _rb_test > 0 and _ft_test > 0:
    gain = _ft_test - _rb_test
    ax.set_title(f'FinRAG — Rule-Based vs LoRA Fine-Tuned  (test improvement: +{gain:.1%})')
else:
    ax.set_title('FinRAG — Rule-Based vs LoRA Fine-Tuned')
ax.set_ylabel('Execution Accuracy')
ax.set_ylim(0, max(accs) * 1.15 + 0.02)
plt.tight_layout()
plt.savefig('outputs/figures/accuracy_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✓ Plots saved → outputs/figures/')

In [ ]:
# ── Cell 9: Save report + final summary ──────────────────────────────────────
import json, os
os.makedirs('outputs', exist_ok=True)

_rb_dev  = report_rb_dev['overall']['accuracy']  if 'report_rb_dev'  in dir() else None
_rb_test = report_rb_test['overall']['accuracy'] if 'report_rb_test' in dir() else None
_ft_dev  = dev_eval['summary']['accuracy']       if 'dev_eval'       in dir() else None
_ft_test = test_eval['summary']['accuracy']      if 'test_eval'      in dir() else None

combined = {
    'dataset_sizes': {'train': len(train_examples), 'dev': len(dev_examples), 'test': len(test_examples)},
    'rule_based':    {'dev': _rb_dev,  'test': _rb_test},
    'finetuned_lora':{'dev': _ft_dev,  'test': _ft_test,
                      'model': MODEL_NAME if 'MODEL_NAME' in dir() else '',
                      'adapter': ADAPTER_DIR if 'ADAPTER_DIR' in dir() else ''},
    'full_report':   report_ft if 'report_ft' in dir() else {},
}
with open('outputs/evaluation_report.json', 'w') as f:
    json.dump(combined, f, indent=2, default=str)

sep = '=' * 60
print(sep)
print('  FINRAG — FINAL RESULTS')
print(sep)
print(f'  Dataset   train {len(train_examples):>5}  |  dev {len(dev_examples):>4}  |  test {len(test_examples):>5}')
print()
print(f'  {"Method":<30} {"Dev":>6}  {"Test":>6}')
print(f'  {"-"*46}')
if _rb_dev  is not None: print(f'  {"Rule-Based (no LLM)":<30} {_rb_dev:>6.1%}  {_rb_test:>6.1%}')
if _ft_test is not None: print(f'  {"LoRA Fine-Tuned (Llama-3.2-3B)":<30} {_ft_dev:>6.1%}  {_ft_test:>6.1%}')
if _rb_test and _ft_test:
    gain = _ft_test - _rb_test
    print(f'  {"Improvement from fine-tuning":<30}          {gain:>+6.1%}')
print()

if 'report_ft' in dir():
    r = report_ft
    print(f'  Program gen rate  : {r["program_induction"].get("program_generation_rate",0):.1%}')
    print(f'  Exec success rate : {r["numerical_reasoning"].get("execution_accuracy",0):.1%}')
    print(f'  Context recall    : {r["context_filtering"].get("mean_recall",0):.1%}')
    print(f'  Context F1        : {r["context_filtering"].get("mean_f1",0):.1%}')
    print(f'  Causal detect     : {r["causality_detection"].get("detection_rate",0):.1%}')
    print(f'  Temporal score    : {r["temporal_reasoning"].get("mean_temporal_score",0):.3f}')
    print()
    print('  Per question type (test, fine-tuned):')
    for qtype, info in r.get('per_type_accuracy', {}).items():
        print(f'    {qtype:16s}: {info["accuracy"]:.1%}  ({info["correct"]}/{info["count"]})')

print()
print(f'  ✓ Report  → outputs/evaluation_report.json')
print(f'  ✓ Figures → outputs/figures/')
print(sep)

In [ ]:
# ── Cell 10: Single-example debug (test split) ───────────────────────────────
# Change EXAMPLE_IDX to inspect any test question.
# Compares rule-based prediction vs fine-tuned prediction side by side.

EXAMPLE_IDX = 0   # ← change to explore other examples

from src.reasoning.numerical_reasoner import NumericalReasoner
from src.pipeline import FinancialQAPipeline
from src.utils.financial_utils import answers_match

ex = test_examples[EXAMPLE_IDX]
nr = NumericalReasoner()

print('━' * 65)
print(f'Question     : {ex.question}')
print(f'Gold answer  : {ex.answer}')
print(f'Gold program : {ex.program_str}')
if ex.table:
    print(f'Table header : {ex.table[0]}')
    if len(ex.table) > 1: print(f'Table row 1  : {ex.table[1]}')
print('━' * 65)

# Rule-based
result_rb = pipeline_rb.answer(ex)
rb_pred   = result_rb.get('predicted_answer', '(empty)')
rb_prog   = result_rb.get('numerical', {}).get('induced_program', [])
print(f'\nRule-based')
print(f'  Program   : {rb_prog}')
print(f'  Prediction: {rb_pred}  {"✓" if answers_match(rb_pred, ex.answer) else "✗"}')

# Fine-tuned
if 'finqa_trainer' in dir() and finqa_trainer.model is not None:
    ft_prog = finqa_trainer._generate_program(ex.question, ex.table, ex.context_text)
    ft_pred = '(empty)'
    if ft_prog:
        steps  = nr.parse_finqa_program(ft_prog)
        res    = nr.execute_program(steps, ex.table)
        if res['success'] and res['result'] is not None:
            ft_pred = FinancialQAPipeline._format_numerical_answer(res['result'])
    print(f'\nLoRA Fine-tuned')
    print(f'  Program   : {ft_prog}')
    print(f'  Prediction: {ft_pred}  {"✓" if answers_match(ft_pred, ex.answer) else "✗"}')

print(f'\nGold answer  : {ex.answer}')